# arms / preference — what the training signal pushes toward, per arm  `[TRAINING]`

**What this family answers.** Inside the training signal of each of the four arms (PTO K=0/K=5, GRPO
K=0/K=5, all on one axis): what, in words, the update pushes the policy toward, how that target moves
across iterations, and **whether it predicts the eval move it produced**. Exports →
`results/arms/preference/{figures,tables}/<judge>/`. Ported from `analysis/6_Preference` §1–§5; the
K=0-vs-K=5 **mechanism chain** (select → generate → evaluate on the over-praise channel, old §5d) now
lives in `lookahead/mechanism` and is not rendered here.

**§1–§2 — PTO's preference pairs** (the original Mass-Mean-Probe over `pref_pairs/pairs.csv`): per
iteration the unit **preference direction** = normalized mean(chosen − rejected) embedding, and
projecting words / MI-concepts onto it reads out the preference. §2 overlays the two PTO look-ahead arms
(descriptive).

**§3 — both methods, one probe.** GRPO has no preference pairs, but "preference" was never the essential
thing: both methods weight the candidates of a group and push the policy along the weighted sum,
differing only in the weights — DPO puts **±1** on the recorded chosen/rejected, GRPO uses the
**standardized group-relative advantage** that actually scales each completion's gradient. Rescaled to
a common per-group size they are directly comparable, so *"what does each method actually reward?"* gets
a like-for-like answer on all four arms.

**§4 — does the signal predict the outcome?** Joins each iteration's update features to the
persona-paired eval delta that iteration produced (`model_iter_n` vs `model_iter_{n-1}`, N=96 personas).

**§5 — three questions the aggregates can't answer.** Is the PTO-vs-GRPO gap the *loss* or the *data*
(swap the weighting rule on fixed groups, at each K)? Does the update *pull* the drift or *follow* it
(what the policy generates vs what the update selects)? And how much usable signal is there at all —
plus the actual text, early vs late.

> **Judge handling.** Every section here reads the TRAINING side (`generations.jsonl` candidate rewards,
> `pairs.csv` roles) — produced by the training oracle (gpt-4o-mini) during the run and impossible to
> re-grade after the fact — so the whole family is saved ONLY under the primary leaf (`gpt-4o-mini/`).
> §4 joins to the eval side, which is grader-dependent in principle; as in the predecessor it uses the
> primary oracle only and says so. Under a held-out `EDA_JUDGE` this notebook prints a pointer and
> renders nothing (a byte-identical copy under another grader's leaf would imply a measurement that never
> happened). The multi-judge work lives in `measurement/validity`.
>
> ⚠️ **§3 also re-measures the probe itself, and the news is not good for §1–§2.** A direction estimated
> from one iteration's PTO pairs has a **split-half cosine of ~0.19** — two halves of the same iteration
> point almost independently — and its held-out win rate is well below the in-sample number §1 reports.
> Read §1's per-iteration drift artifacts (word drift, learn/unlearn, MI-concept curves, direction drift)
> as **mostly estimation noise**; the claims that survive are the ones §3 and §5 make from *pooled*
> directions and from the exact, embedding-free lexical contrasts.

In [ ]:
import sys, os
_p = os.path.abspath(".")                      # find eda/ (the dir holding eda_analysis/) from any depth
while _p != os.path.dirname(_p) and not os.path.isdir(os.path.join(_p, "eda_analysis")):
    _p = os.path.dirname(_p)
sys.path.insert(0, _p)
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
pd.set_option("display.width", 185, "display.max_columns", 50)

import os, eda_analysis
from eda_analysis import exports, plotting
from eda_analysis import training, pref
from eda_analysis.constants import judge_dirname
cfg = eda_analysis.EdaConfig(family="arms/preference", judge=os.environ.get("EDA_JUDGE", ""))
S = eda_analysis.notebook_setup(cfg)
exports.reset_results()   # clears only THIS family's generated figures/tables for the active judge (never SUMMARY.md)

# The whole family is TRAINING-side (generations.jsonl candidate rewards + pairs.csv roles, produced by
# the training oracle gpt-4o-mini during the run) — not judge-swappable, so it renders under the primary
# leaf only. §4's eval side is the primary oracle too (as in the predecessor).
TRAINING_SIDE = (S.JUDGE == "")
GRADER = judge_dirname(S.JUDGE)
PRIMARY_LEAF = f"results/{S.FAMILY}/{{figures,tables}}/{judge_dirname('')}/"
CENSOR = "GRPO K=5 stopped after iteration 5 (censored; its training rows end at train_iter 5)."
TRAIN_ORACLE = "training oracle gpt-4o-mini (partial-conv candidate rewards; not judge-swappable)"
NUM = {}                                       # number ledger, filled per section, saved at the end
if TRAINING_SIDE:
    exports.save_provenance(cfg, S.SCORES)     # re-stamp the banner reset_results just removed
else:
    print(f"[arms/preference] TRAINING-side family — judge-invariant by construction; saved under the "
          f"primary leaf only ({PRIMARY_LEAF}). Skipped for EDA_JUDGE={S.JUDGE!r} ({GRADER}).")
FOCUS = sorted(S.SCORES.arm.unique()) if not S.SCORES.empty else [a.label for a in S.ARMS]
print("TRAINING_SIDE =", TRAINING_SIDE, "| GRADER =", GRADER, "| arms =", [a.label for a in S.ARMS])

## 0 · Arms with training data
PTO arms with preference pairs drive §1–§2; §3–§5 pick up every arm of both methods from `generations.jsonl`.

In [ ]:
PTO_ARMS = []
if TRAINING_SIDE:
    PTO_ARMS = [a for a in S.ARMS if a.method == "PTO" and len(training.load_pref_pairs([a]))]
    print("PTO arms with preference pairs (sections 1-2):", [a.label for a in PTO_ARMS])
    print("all arms (sections 3-5):", [a.label for a in S.ARMS])

## 1 · Per-arm preference, iteration by iteration  `[TRAINING]`
**Purpose.** Per PTO arm: probe quality (does the direction separate the pairs? wins > 0.5) → pooled
word ranking → per-iteration word drift (heatmap + top-words table) → **direction drift in 2D** +
consecutive cosine → **learned vs unlearned words** → MI-concept drift + a first→last read-out.
Artifacts are named `<arm>_pref_*` (one set per PTO arm).

> ⚠️ **`wins_correct` here is IN-SAMPLE** — the direction is scored on the very pairs it was fitted on,
> and at PTO's ~400 pairs an iteration that inflates it by ~0.13. §3 reports both, plus the split-half
> cosine that says these per-iteration directions are only ~0.19 reliable. Everything in this section is
> therefore **more noise than it looks**; the pooled, audited versions are in §3.

In [ ]:
RESULTS = {}   # arm -> {DIRS, CAT} for the K0-vs-K5 overlay in §2 and the sanity gate in §3

def analyze_pref(arm):
    PAIRS = pref.add_text_features(training.load_pref_pairs([arm]))
    if PAIRS.empty:
        print(f"{arm.label}: no pairs."); return
    EMB = pref.embed_pairs(PAIRS)
    DIRS = pref.preference_direction_by_iter(EMB)
    print(f"\n################  {arm.label}  ################")
    PQ = pref.probe_quality_by_iter(EMB, DIRS)
    print("[probe] wins_correct should be > 0.5 for a real preference axis:"); display(PQ.round(4))
    exports.save_table(PQ.round(4), f"{arm.label}_pref_probe_quality",
                       caption=f"{arm.label} preference-probe quality per training iteration (wins_correct, gap, margin) "
                               f"over the tau-filtered pairs.csv pairs; unit = one emitted pair; {TRAIN_ORACLE}. "
                               f"wins_correct is IN-SAMPLE (optimistic) -- the honest wins_holdout is in update_direction_quality.")
    words, wmat = pref.embed_vocab(pref.build_vocab(PAIRS, top_n=3000)); WP = pref.word_projection(words, wmat, DIRS)
    # overall preference + per-iteration word drift
    fig = pref.pref_word_ranking(WP, title=f"{arm.label}: words by preference projection (green=chosen, red=rejected)")
    if fig: exports.save_fig(fig, f"{arm.label}_pref_word_ranking",
                             caption=f"{arm.label} top chosen/rejected-aligned words (Mass Mean Probe, pooled over training "
                                     f"iterations; + = chosen-aligned); {TRAIN_ORACLE}."); plt.show()
    fig = pref.pref_word_drift_heatmap(WP, title=f"{arm.label}: preferred-word drift across iterations")
    if fig: exports.save_fig(fig, f"{arm.label}_pref_word_drift",
                             caption=f"{arm.label} per-iteration projection of the top chosen-/rejected-aligned words (drift; "
                                     f"per-iteration directions, see the reliability caveat -- mostly estimation noise); {TRAIN_ORACLE}."); plt.show()
    display(pref.top_words_by_iter(WP, k=8))
    # direction drift (vectors) + learned/unlearned words
    fig = pref.plot_direction_drift(pref.preference_direction_drift(DIRS), title=f"{arm.label}: preference-direction drift")
    if fig: exports.save_fig(fig, f"{arm.label}_pref_direction_drift",
                             caption=f"{arm.label} preference direction in 2D PCA + consecutive cosine (how the preferred axis "
                                     f"re-orients across training iterations; per-iteration directions, split-half ~0.19, so read "
                                     f"as noise-dominated); {TRAIN_ORACLE}."); plt.show()
    fig = pref.plot_learn_unlearn(pref.learn_unlearn_words(WP, k=8))
    if fig: exports.save_fig(fig, f"{arm.label}_pref_learn_unlearn",
                             caption=f"{arm.label} words most newly preferred (learned) vs dropped (unlearned) across "
                                     f"consecutive training-iteration transitions; {TRAIN_ORACLE}."); plt.show()
    # MI-concept drift + first->last read-out
    CAT = pref.category_projection(DIRS)
    if not CAT.empty:
        fig = pref.plot_category_drift(CAT)
        if fig: exports.save_fig(fig, f"{arm.label}_pref_category_drift",
                                 caption=f"{arm.label} MI-concept word groups projected onto the chosen-rejected direction per "
                                         f"training iteration (+ = more preferred); {TRAIN_ORACLE}."); plt.show()
        exports.save_table(CAT.round(4), f"{arm.label}_pref_MI_concepts",
                           caption=f"{arm.label} MI-concept projection onto the preference direction per training iteration "
                                   f"(+ = the concept's words are chosen-aligned); {TRAIN_ORACLE}.")
        wide = CAT.pivot_table(index="category", columns="train_iter", values="score")
        f0, fl = wide.columns.min(), wide.columns.max()
        print(f"[MI-concept shift iter {f0}->{fl}; + = MORE preferred over training]:")
        for cat, d in (wide[fl] - wide[f0]).sort_values(ascending=False).items():
            print(f"   {cat:<16} {wide.loc[cat, f0]:+.3f} -> {wide.loc[cat, fl]:+.3f}   (Δ {d:+.3f})")
    RESULTS[arm.label] = {"DIRS": DIRS, "CAT": CAT}

if TRAINING_SIDE:
    for arm in PTO_ARMS:
        analyze_pref(arm)
    if not PTO_ARMS:
        print("No PTO arm with preference pairs scored yet.")

## 2 · Does look-ahead change *what* is preferred? K=0 vs K=5 (PTO)  `[TRAINING]`

> ⚠️ **DESCRIPTIVE only.** This K=0-vs-K=5 overlay is **hypothesis-generating**, not an inferential K
> test — the tested look-ahead contrasts live in `lookahead/*`, and the mechanism chain on the over-praise
> channel (old §5d) is `lookahead/mechanism`. Per-iteration directions, so the §3 reliability caveat applies.

**Purpose.** Overlay PTO_LA0 vs PTO_LA5 MI-concept preference across iterations, and the cosine between
their per-iteration directions. **Read:** diverging curves / low cosine = look-ahead *suggests* steering the
preference toward different language.

In [ ]:
if TRAINING_SIDE:
    if {"PTO_LA0", "PTO_LA5"} <= set(RESULTS):
        fig = pref.plot_category_compare({a: RESULTS[a]["CAT"] for a in ("PTO_LA0", "PTO_LA5")},
                                         palette=S.PALETTE, title="MI-concept preference: K=0 vs K=5 (PTO)")
        if fig:
            exports.save_fig(fig, "pref_category_K0_vs_K5",
                             caption="PTO MI-concept preference projection across training iterations, K=0 vs K=5 -- does "
                                     "look-ahead change what the policy prefers? Descriptive overlay of per-iteration "
                                     "pairs.csv directions (see the reliability caveat and the measured split-half in section 3); "
                                     f"{TRAIN_ORACLE}. The tested K contrasts are in lookahead/*."); plt.show()
        d0, d5 = RESULTS["PTO_LA0"]["DIRS"], RESULTS["PTO_LA5"]["DIRS"]
        common = sorted(set(d0) & set(d5))
        if common:
            print("cos(dir_K0, dir_K5) at matched iters:", {i: round(float(d0[i] @ d5[i]), 3) for i in common})
            print("  ^ uncorrected for estimation noise — §3 reports the attenuation-corrected version.")
    else:
        print(f"Need both PTO_LA0 and PTO_LA5 with pref pairs for the K overlay (have: {sorted(RESULTS)}).")

## 3 · One probe, both methods — what does each update actually reward?  `[TRAINING]`

**Purpose.** Put PTO and GRPO on the same axes, all four arms. Every candidate carries the weight its
method's update gives it (DPO's ±1 chosen/rejected; GRPO's standardized advantage), rescaled per group to
a common size, so a "unit of push" means the same thing on both sides. Two complementary readouts:

- **Lexical push** (`weighted_lexical_contrast`) — exact, no embeddings, **every** group: Σ w·feature per
  group, so `+40` on length = "the update pushes toward ~40-character-longer completions" and `0` =
  indifferent. Comes with a standard error, because these are small per-pair numbers.
- **Semantic direction** (`direction_by_iter` / `direction_by_arm`) — `normalize(Σ w · embedding)`, the
  generalization of §1's Mass-Mean-Probe to any weighting. Everything downstream (word projection, MI
  concepts) takes a plain direction, so it works unchanged for GRPO.

**And the probe gets audited.** `direction_quality` reports three numbers §1 never had: `wins_holdout`
(each half judged by the *other* half's direction — the honest version of §1's in-sample `wins_correct`),
`split_half_cos` (is the direction even estimated?), and, for cross-arm cosines, the **attenuation
ceiling** — two noisy directions cannot correlate to 1 even if identical, so a raw cosine means nothing
without it. Same correction `measurement/validity` applies to cross-judge agreement, for the same reason.

**Sampling.** Directions cap at 400 groups per (arm, iteration), seeded per (arm, iteration) with
`BOOT_SEED` — chosen by measuring reliability at 50/100/200/400, not guessed. The lexical half uses everything.

In [ ]:
if TRAINING_SIDE:
    CANDS = pref.load_weighted_candidates(S.ARMS)
    print("arms with weighted training candidates:", sorted(CANDS.arm.unique()) if not CANDS.empty else "none")

    # ── 3a · the exact, embedding-free half: what the update pushes toward, per iteration ──
    LEX = pref.weighted_lexical_contrast(CANDS)
    display(LEX.round(4))
    exports.save_table(LEX.round(4), "update_lexical_push",
                       caption="Per (arm, training iteration), all four arms: the lexical contrast the update pushes for, "
                               "Sum(w*feature) per group +/- SE, on a shared scale for both methods (DPO's +/-1 pair; GRPO's "
                               "standardized advantages rescaled to match). + = the update favours MORE of the feature; 0 = "
                               f"indifferent. Uses EVERY gradient group (no embedding, no sampling); {TRAIN_ORACLE}. {CENSOR}")
    fig = pref.plot_lexical_push(LEX, palette=S.PALETTE)
    if fig: exports.save_fig(fig, "update_lexical_push",
                             caption="What each training iteration's update pushes toward, all four arms on one scale: completion "
                                     "length, question marks, affirmation and over-praise markers (+/-1 SE over groups; + = "
                                     f"favours more of the feature); {TRAIN_ORACLE}. {CENSOR}"); plt.show()

    # ── 3b · the semantic half + the probe audit ──────────────────────────────────────
    EMB = pref.embed_candidates(pref.sample_groups(CANDS))
    DIRS = pref.direction_by_iter(EMB)
    QUAL = pref.direction_quality(EMB, DIRS)
    print("[probe audit] wins_holdout is the honest win rate; split_half_cos < ~0.5 means the "
          "per-iteration direction is not yet measured, whatever the projections show:")
    display(QUAL.round(3))
    exports.save_table(QUAL.round(4), "update_direction_quality",
                       caption="Per (arm, training iteration) probe audit of the update direction, all four arms: in-sample "
                               "wins_correct vs the honest wins_holdout (each half scored by the other half's direction), mean "
                               "projection gap, and split_half_cos = cosine between directions fitted on disjoint halves (the "
                               "precision check; halves drawn with BOOT_SEED per (arm, iteration)). A low split_half_cos means the "
                               f"direction is real but unmeasured at this group count, NOT that the update has no target; {TRAIN_ORACLE}. {CENSOR}")

    DIRS_ARM = pref.direction_by_arm(EMB)
    QUAL_POOLED = pref.pooled_direction_quality(EMB, DIRS_ARM)
    display(QUAL_POOLED.round(3))
    exports.save_table(QUAL_POOLED.round(4), "update_direction_quality_pooled",
                       caption="The same audit for the per-ARM direction pooled over all training iterations, all four arms -- the "
                               "estimate cross-method claims should rest on, since pooling multiplies the group count and the "
                               f"split-half cosine rises accordingly; {TRAIN_ORACLE}. {CENSOR}")

    COS = pref.pooled_direction_cosines(DIRS_ARM, QUAL_POOLED)
    print("[pooled direction cosines] read `cosine_corrected` — `cosine` alone is capped by how well "
          "each direction is estimated (`ceiling`):")
    display(COS.round(3))
    exports.save_table(COS.round(4), "update_direction_cosines",
                       caption="Cosine between every pair of arms' POOLED update directions (all six arm pairs), with the "
                               "attenuation ceiling sqrt(r_a*r_b) from each direction's Spearman-Brown-corrected split-half "
                               "reliability and the corrected cosine. 1.0 corrected = the two updates pull toward the same "
                               f"language; the ceiling is why a raw cosine of 0.3 is not interpretable on its own; {TRAIN_ORACLE}.")

    # ── 3c · MI concepts, both methods on one figure ──────────────────────────────────
    CATS = {arm: pref.category_projection(d) for arm, d in DIRS.items()}
    fig = pref.plot_category_compare(CATS, palette=S.PALETTE,
                                     title="MI-concept preference by arm — both methods, same probe")
    if fig: exports.save_fig(fig, "update_category_by_arm",
                             caption="Each MI-concept word group projected onto the per-iteration update direction (+ = the "
                                     "update favours that concept's words), one line per arm, all four arms. Per-iteration "
                                     "directions: read alongside update_direction_quality, whose split_half_cos says how much of "
                                     f"each curve is estimation noise; {TRAIN_ORACLE}. {CENSOR}"); plt.show()

    # ── 3d · sanity gate: does the candidate-derived PTO direction match pairs.csv? ────
    for arm_label, r in RESULTS.items():
        AG = pref.direction_agreement_with_pairs(EMB, arm_label, r["DIRS"])
        if not AG.empty:
            print(f"[sanity] {arm_label}: cos(candidate-derived, pairs.csv-derived) per iter = "
                  f"{dict(zip(AG.train_iter, AG.cosine.round(3)))}")
            print(f"         mean {AG.cosine.mean():.3f} — two independent logs of the same DPO update; "
                  "well below 1 would be a data-integrity problem, not a plotting one.")
            NUM[f"sanity.{arm_label}.mean_cos_candidates_vs_pairs"] = {
                "value": float(AG.cosine.mean()), "source": "(printed in section 3d)",
                "note": "cosine between the generations.jsonl-derived and pairs.csv-derived per-iteration DPO directions, mean over iterations"}

    for _, r in QUAL_POOLED.iterrows():
        NUM[f"probe.{r.arm}.pooled_wins_holdout"] = {"value": float(r.wins_holdout), "source": "tables/update_direction_quality_pooled.md",
                                                     "note": "held-out win rate of the per-arm pooled update direction"}
        NUM[f"probe.{r.arm}.pooled_split_half_cos"] = {"value": float(r.split_half_cos), "source": "tables/update_direction_quality_pooled.md",
                                                       "note": "split-half cosine of the per-arm pooled update direction"}
    for _, r in COS.iterrows():
        NUM[f"cosine.{r.arm_a}_vs_{r.arm_b}.corrected"] = {"value": float(r.cosine_corrected), "source": "tables/update_direction_cosines.md",
                                                            "note": "attenuation-corrected cosine between the two arms' pooled update directions"}

## 4 · Does the training signal predict the eval move?  `[TRAINING] → [EVAL]`

**Purpose.** Everything above describes what the update *wanted*; this asks whether it explains what the
model *did*. Each iteration's features are joined to the persona-paired eval delta that same update
produced — update in **train_iter n** → adapter `iteration_n` → conversations `model_iter_n`, so its effect
is `eval(model_iter_n) − eval(model_iter_{n-1})`, paired over the 96 shared personas (never a difference of
iteration means: the personas are reshuffled every iteration).

**Read `rho_partial_iter`, not `spearman_rho`.** Nearly every feature here rises monotonically over
training, and the eval deltas trend too (gains taper, MICI grows) — so *any* monotone feature correlates
with *any* monotone delta, and the raw ρ is confounded with iteration index almost by construction. The
partial removes `train_iter` from both sides: what survives it is "the iterations that pushed harder moved
further **relative to where they sat in training**".

> ⚠️ **Correlational, n ≤ 10 per arm, uncorrected across the feature × metric grid.** With ~4 features ×
> several metrics × 4 arms, a few rows at *p* < .05 are expected by chance — a *pattern* across related
> features within one arm is worth something, an isolated star is not. It can also never separate "the
> update pushed affirmation, which raised MICI" from "iterations where the policy was drifting anyway also
> had affirmation-heavy branches". `<METHOD> (pooled)` rows pool that method's K=0 and K=5 arms.
>
> The eval side is the **primary oracle only** (this family renders under the primary leaf; the training
> signal cannot be re-graded). The cross-judge robustness of the MICI outcome itself lives in
> `measurement/validity`.

In [ ]:
if TRAINING_SIDE:
    FEATS = pref.preference_features_by_iter(CANDS, directions=DIRS, quality=QUAL)
    LINK = pref.link_to_outcomes(FEATS, S.SCORES, metrics=S.METRICS)
    if LINK.empty:
        print("no iteration has both its training signal and both eval endpoints scored.")
    else:
        key = ["arm", "train_iter", "metric", "n_groups", "w_affirm", "w_question", "w_len",
               "w_overpraise", "delta_mean", "dz", "p"]
        display(LINK[[c for c in key if c in LINK.columns]].round(4))
        exports.save_table(LINK.round(4), "pref_outcome_link",
                           caption="Per (arm, train_iter, metric), all four arms: every feature of that iteration's update next to "
                                   "the persona-paired eval delta it produced (model_iter_n minus model_iter_n-1; delta_mean/dz/p "
                                   "from compare_two_models, paired on persona_id, N=96; + = the later policy scores higher, MICI "
                                   "lower = better). The training-signal -> eval-move join. Training features from the training "
                                   f"oracle gpt-4o-mini; eval side graded by the primary oracle only. {CENSOR}")

        CORR = pref.outcome_correlations(LINK)
        top = CORR.reindex(CORR.rho_partial_iter.abs().sort_values(ascending=False).index)
        print("=== strongest links AFTER partialling out train_iter (the column to read) ===")
        display(top.head(15).round(3))
        exports.save_table(CORR.round(4), "pref_outcome_correlations",
                           caption="Spearman rho between each update feature and the eval delta it preceded, per arm and pooled per "
                                   "method (a pooled row pools that method's K=0 and K=5 arms). spearman_rho is the raw association; "
                                   "rho_feature_vs_iter shows how much the feature simply trends with training; rho_partial_iter is "
                                   "the one to read (train_iter partialled out of both sides). Descriptive: n <= 10 iterations per "
                                   "arm, no multiplicity correction; unit = one training iteration; eval side primary oracle only.")

        for feat in ["w_affirm", "w_overpraise", "w_question"]:
            fig = pref.plot_pref_outcome(LINK, feature=feat, palette=S.PALETTE)
            if fig:
                exports.save_fig(fig, f"pref_outcome_{feat}",
                                 caption=f"Update feature `{feat}` (what the training reward selected for, training oracle "
                                         f"gpt-4o-mini) against the persona-paired eval delta of the same iteration (primary "
                                         f"oracle, N=96 personas), one point per training iteration (label = iteration), per arm, "
                                         f"all four arms. Dashed = per-arm least squares, corner text = raw Spearman rho; the "
                                         f"partial-correlation version is in pref_outcome_correlations. {CENSOR}")
                plt.show()

## 5 · Three questions the aggregate curves can't answer  `[TRAINING]`

**5a · Is "PTO vs GRPO" about the loss, or about the data?** The as-trained direction cosine confounds
them: the two methods see different candidate **pools** *and* apply different weighting **rules**. Every
group logs all its candidates' scores, so `reweight` can hold the groups fixed and swap only the rule —
giving three rows to compare: as trained, same-data/other-rule, and same-rule/other-data. If the same-rule
rows stay as low as the as-trained one while the same-data rows are high, then the methods diverge because
of **what they generate**, not **how they weight it** — and the thesis comparison is a statement about the
candidate pool rather than about DPO vs group-relative PPO. Done once per matched-K pair (K=0, K=5).

**5b · Does the update pull the drift, or follow it?** §3's contrast measures what the update *selects
for within a group*. It says nothing about what the policy *generates* in the first place. Plotting the
pool mean above the selection contrast separates them — and the gap between the two rows is the whole
reward-hacking mechanism in one figure.

**5c · How much usable signal is there, and what does it look like?** GRPO trains on every prompt it
builds; PTO emits a pair only where the best and worst branch differ by more than τ. If PTO's yield falls
as its branches converge, its later iterations are training on less — a candidate explanation for a
flattening curve that no outcome figure can see. Then, because every other artifact here is an aggregate:
the actual text of the most decisive pairs, early vs late.

In [ ]:
if TRAINING_SIDE:
    CANDS_ALL = pref.load_weighted_candidates(S.ARMS, drop_zero_weight=False)   # every candidate, incl. w=0

    # ── 5a · loss or data? Same groups, swapped rule — one block per matched-K pair ──
    METHOD_ARMS = {m: sorted({a.label for a in S.ARMS if a.method == m}) for m in ("PTO", "GRPO")}
    pairs = [(p, g) for p in METHOD_ARMS["PTO"] for g in METHOD_ARMS["GRPO"]
             if p.split("_LA")[1] == g.split("_LA")[1]]           # matched K only
    if pairs:
        EMB_ALL = pref.embed_candidates(pref.sample_groups(CANDS_ALL))
        blocks = []
        for arm_p, arm_g in pairs:
            K = int(arm_p.split("_LA")[1])
            # per-arm quantities are independent of the other arms in the frame, so subsetting is exact
            DEC = pref.weighting_decomposition(EMB_ALL[EMB_ALL.arm.isin([arm_p, arm_g])], arm_p, arm_g)
            print(f"=== K={K}: is the {arm_p} vs {arm_g} divergence the LOSS or the DATA? ===")
            display(DEC.round(3))
            blocks.append(DEC.assign(K=K)[["K"] + list(DEC.columns)])
            chk = pref.rule_reconstruction_check(CANDS_ALL, arm_p)
            if chk:
                print(f"[sanity] score-only 'dpo' rule vs {arm_p}'s recorded roles: picks a maximum "
                      f"{chk['chosen_picks_a_maximum']:.3f} / a minimum {chk['rejected_picks_a_minimum']:.3f} "
                      f"of the time; {chk['tie_rate_at_max']:.1%} of groups have TIED maxima, so exact row "
                      "identity is not the right test — the direction cosine above is.")
            row = DEC[DEC.comparison.str.startswith("as trained")]
            if len(row):
                NUM[f"decomposition.K{K}.as_trained_cosine_corrected"] = {
                    "value": float(row.cosine_corrected.iloc[0]), "source": "tables/weighting_decomposition.md",
                    "note": f"{arm_p} vs {arm_g} pooled update directions, attenuation-corrected"}
        DEC_ALL = pd.concat(blocks, ignore_index=True)
        exports.save_table(DEC_ALL.round(4), "weighting_decomposition",
                           caption="Cosine between update directions under three conditions, one block per matched-K PTO/GRPO pair "
                                   "(K = 0 and K = 5): as trained (rule AND data differ), same groups with the rule swapped (the "
                                   "RULE's effect), and same rule on each method's own groups (the DATA's effect). Read the `read` "
                                   "column: the attenuation ceiling assumes independent estimation error, which holds across arms "
                                   "but NOT for the same-groups rows, where both directions share their noise and the correction "
                                   f"over-corrects (values above 1.0 are the tell). Pooled over training iterations; {TRAIN_ORACLE}. {CENSOR}")
    else:
        print("no matched-K PTO/GRPO pair among the arms — the loss-vs-data decomposition needs both.")

    # ── 5b · generation vs selection ─────────────────────────────────────────────────
    POOL = pref.pool_mean_by_iter(CANDS_ALL)
    display(POOL.round(3))
    exports.save_table(POOL.round(4), "generation_pool_means",
                       caption="Per (arm, training iteration), all four arms: the mean of each lexical feature over ALL candidates "
                               "-- what the policy GENERATES, as opposed to what the update selects for. NOTE the indexing: "
                               "train_iter n samples from the iter-start policy, so this row describes the same policy the eval "
                               f"set calls model_iter_{{n-1}}; {TRAIN_ORACLE}. {CENSOR}")
    fig = pref.plot_selection_vs_generation(POOL, LEX, palette=S.PALETTE)
    if fig: exports.save_fig(fig, "generation_vs_selection",
                             caption="Top row: what the policy produces (unweighted mean over every candidate). Bottom row: what the "
                                     "update pushes toward within a group (+ = favours more of the feature). All four arms. A "
                                     "large gap in scale between the rows means the per-step selection pressure is small while "
                                     "the generated distribution moves a long way -- a compounding on-policy loop rather than one "
                                     f"hard pull; {TRAIN_ORACLE}. {CENSOR}"); plt.show()

    # ── 5c · how much signal, and what does it look like ─────────────────────────────
    YIELD = pref.pair_yield_by_iter(S.ARMS)
    display(YIELD.round(3))
    exports.save_table(YIELD.round(4), "training_signal_yield",
                       caption="Per (arm, training iteration), all four arms: groups_built (branch points / prompt groups logged), "
                               "groups_trained (those with both an up- and a down-weighted side), yield_rate, and the best-worst "
                               "training-reward gap. GRPO trains on essentially every group; PTO's tau filter drops the branch "
                               f"points whose branches tie; {TRAIN_ORACLE}. {CENSOR}")
    fig = pref.plot_pair_yield(YIELD, palette=S.PALETTE)
    if fig: exports.save_fig(fig, "training_signal_yield",
                             caption="How much usable training signal each iteration produced, all four arms: groups that trained, "
                                     f"the yield rate, and how decisive those groups were (best-worst training-reward gap); {TRAIN_ORACLE}. {CENSOR}"); plt.show()
    for arm, g in YIELD.groupby("arm"):
        NUM[f"yield.{arm}.min_yield_rate"] = {"value": float(g.yield_rate.min()), "source": "tables/training_signal_yield.md",
                                              "note": "lowest per-iteration share of built groups that produced a gradient"}
        NUM[f"yield.{arm}.groups_trained_total"] = {"value": int(g.groups_trained.sum()), "source": "tables/training_signal_yield.md",
                                                    "note": "gradient groups summed over training iterations"}

    for arm_label in sorted(CANDS.arm.unique()):
        its = sorted(CANDS[CANDS.arm == arm_label].train_iter.unique())
        EX = pref.pref_examples(CANDS, arm=arm_label, iters=[its[0], its[-1]], k=3)
        if not EX.empty:
            print(f"\n=== {arm_label}: most decisive up- vs down-weighted completions, first vs last iteration ===")
            display(EX)
            exports.save_table(EX, f"{arm_label}_examples",
                               caption=f"{arm_label}: the 3 groups per iteration with the largest training-reward gap, first and "
                                       f"last training iteration, as text ({TRAIN_ORACLE}). Illustration of what the update was "
                                       f"choosing between -- deliberately the most decisive groups, NOT a random sample, so read "
                                       f"them as exhibits and never as a rate.")

## 6 · How to read this family
- **Is the probe real?** Read **`wins_holdout`** (§3), not §1's `wins_correct` — the latter scores the
  direction on the same groups it was fitted on and is optimistic by ~0.13 at PTO's group counts.
- **Is the probe *measured*?** `split_half_cos` (§3). Below ~0.5 the direction is not pinned down at that
  group count, so per-iteration projections built on it are mostly noise however smooth the curve looks.
  This is why §1–§2's drift artifacts carry the header's caveat and why cross-arm claims use the **pooled**
  direction.
- **What / how it drifts:** the word ranking + drift heatmap + learn/unlearn + MI-concept read-out test
  whether **affirmation/achievement** language becomes more preferred while **questions/reflection** fade
  — the latent-space signature of the behaviour drift in `arms/questionnaires` (MITI section). The exact,
  assumption-free version of the same test is §3's **lexical push**, which needs no embedding and uses
  every group.
- **Do the two methods want the same thing?** §3's `update_direction_cosines` (`cosine_corrected`) says
  *how much* they differ; §5a's `weighting_decomposition` says *why*, at each K — swap the weighting rule
  on fixed groups and see whether the gap closes.
- **Is the reward pulling, or is the policy running?** §5b. The top row (what the policy generates) and
  the bottom row (what the update selects for) are on wildly different scales — small persistent selection
  pressure compounding through an on-policy loop, not one hard pull. Note the off-by-one: `train_iter n`'s
  pool describes the policy the eval set calls `model_iter_{n-1}`.
- **Is there enough signal to train on?** §5c. PTO's τ filter can starve late iterations; GRPO's yield
  stays ~1.0 by construction.
- **Did wanting it work?** §4. Read `rho_partial_iter`; treat everything there as descriptive.
- **Does look-ahead intervene at the reward?** Not here — the select → generate → evaluate chain on the
  over-praise channel, K=0 vs K=5, is `lookahead/mechanism`; §2 is only the descriptive PTO overlay.
- **Caveat:** §1–§3 and §5 are the **[TRAINING]** signal (what the loss optimises), not the eval. Whether
  a shift is *good* is an eval/behaviour question (`arms/outcomes` / `arms/validity` / `arms/stats`); §4 is
  the only place the two sides meet, and it meets them correlationally. GRPO K=5 is censored at iteration 5
  throughout.

The ledger `tables/gpt-4o-mini/preference_numbers.json` collects the per-arm anchors of this family
(pooled probe audit, corrected cross-arm cosines, the as-trained decomposition cosine per K, yield).

In [ ]:
if TRAINING_SIDE and NUM:
    exports.save_numbers("preference_numbers", NUM,
                         caption="Number ledger for arms/preference: per-arm pooled probe audit (wins_holdout, split_half_cos), "
                                 "attenuation-corrected cosines between the arms' pooled update directions, the as-trained "
                                 "PTO-vs-GRPO decomposition cosine per K, the pairs.csv sanity cosine, and the training-signal "
                                 f"yield anchors. Training side only ({TRAIN_ORACLE}); primary leaf only. {CENSOR}")
    print(f"ledger: {len(NUM)} keys")

In [ ]:
exports.prune_orphan_captions(); print("index ->", exports.build_index())